# Parallel computing — Interpreted vs compiled · **Python side**

*Python / Colab track — analogue of the Julia notebook `1_compilation_and_types.jl`.*

We **actually measure** what the Julia notebook only quotes: the pure Python loop,
`numpy`, and a JIT (`numba`). Note the numbers down to compare in class with Julia's `@btime`.

> On Colab: `Runtime ▸ Run all`. The 1st `numba` cell compiles (slow once).

In [ ]:
import timeit

def best_time(fn, repeat=7):
    """Time per call (s), %timeit-style: auto-scale then min over `repeat` trials."""
    number = 1
    while timeit.timeit(fn, number=number) < 0.05:
        number *= 10
    return min(timeit.repeat(fn, number=number, repeat=repeat)) / number

def ms(t): return f"{t*1e3:.3f} ms"
def us(t): return f"{t*1e6:.2f} µs"


In [ ]:
import numpy as np
N = 10_000_000
v = np.random.rand(N)        # numpy array (contiguous, typed)
v_list = v.tolist()          # the SAME data as a pure Python list
len(v_list), type(v_list[0])

## 1. Differences of several orders of magnitude

The same computation — summing 10 million numbers — written three ways.

In [ ]:
def mysum(x):
    s = 0.0
    for xi in x:
        s += xi
    return s

# ⚠️ PURE Python loop over a list: this is the slow case
t_loop = best_time(lambda: mysum(v_list))
print("pure Python loop :", ms(t_loop))

In [ ]:
# numpy: the loop runs in compiled C, not in Python
t_np = best_time(lambda: np.sum(v))
print("numpy.sum        :", ms(t_np))
print(f"\nnumpy is ~{t_loop/t_np:.0f}× faster than the pure Python loop")

**What we observe.** The pure Python loop takes ~hundreds of ms to ~1 s; `np.sum` ~a few ms.

`numpy` is fast because it **does not run the loop in Python**: it calls a compiled C
routine over a contiguous typed array. As soon as you put a **Python loop around the elements**,
you pay the interpreter again on every iteration.

## 2. The CPython *bytecode*

The Python analogue of Julia's `@code_lowered` / `@code_native` is `dis`: we see the **bytecode**
that the CPython virtual machine runs in its interpretation loop — never native code.

In [ ]:
import dis
def f(x):
    return 2*x + 1
dis.dis(f)

`BINARY_OP` looks like a **generic** call: CPython would have to *rediscover* the operand types at
run time and find the right method (`__mul__`, `__add__`…) on **every** operation, on **every** loop
iteration.

But that is only half the story — and the half you're shown is out of date. Since **3.11**
([PEP 659](https://peps.python.org/pep-0659/)), CPython has a *specializing adaptive interpreter*.
It watches the types actually flowing through each bytecode and **rewrites the instruction in place**
into a type-specialized variant. `dis.dis()` hides this: it defaults to `adaptive=False`, so it shows
you the code *before* the interpreter warmed up. Let's ask for the truth.

In [ ]:
def hot_loop(v):
    s = 0.0
    for x in v:
        s += x
    return s

data = [1.0] * 1000
for _ in range(100):                 # warm up: let the interpreter observe the types
    hot_loop(data)

for ins in dis.get_instructions(hot_loop, adaptive=True):
    print(f"  {ins.opname:<24} {ins.argrepr}")

Look at what came back: `FOR_ITER_LIST` and **`BINARY_OP_ADD_FLOAT`**. CPython noticed the values
were floats and specialized the bytecode to them. There is no `__add__` lookup left in this loop.

**So "Python is slow because it can't specialize" is simply false — and the real story makes the
point better.** CPython specializes, but:

| | CPython 3.11+ | Julia |
|---|---|---|
| unit of specialization | **one bytecode** at a time | the **whole function** |
| when | after warm-up, in place | on the 1st call with those types |
| the type check | a **guard on every execution** (wrong type → deoptimize) | **eliminated** — proved once |
| what runs | still the **interpreter loop** | **machine code** |
| the data | still **boxed** `PyObject*` on the heap | unboxed `Float64` in a register |

`BINARY_OP_ADD_FLOAT` still dispatches through the eval loop, still checks that both operands really
are floats, still unboxes two heap objects to add them, and still boxes the result. It removed the
method lookup — the cheapest part. Julia removes the interpreter, the guard **and** the boxing.

> **Key distinction.** It was never "Python can't specialize". It's *specialize **one instruction**,
> keep the guard, stay in the interpreter, stay boxed* vs *specialize **the function**, drop the
> guard, emit native code, unbox*. **That's why the ~100× survives PEP 659** — and why `numba`
> (next section) exists at all.

Julia, for comparison: `f(3.0)` and `f(3)` compile to different native code, and the type check
**disappears** from the hot path. We'll meet this distinction again for **dispatch**, **threads**
and the **GPU**.

## 3. Giving Python a JIT: `numba`

`numba.njit` compiles certain numeric functions to native code — exactly Julia's mechanism,
added *on top of* Python for a subset of the language.

In [ ]:
try:
    from numba import njit
except ImportError:
    !pip -q install numba
    from numba import njit

@njit
def sum_numba(x):
    s = 0.0
    for i in range(x.shape[0]):
        s += x[i]
    return s

sum_numba(v)                         # 1st call: compiles (slow, once)
t_nb = best_time(lambda: sum_numba(v))
print("numba njit loop  :", ms(t_nb))
print(f"≈ {t_loop/t_nb:.0f}× faster than the pure Python loop")

The `numba` loop reaches the order of magnitude of `numpy`/Julia. Lesson: "JIT" is not magic,
it's **specialize by types then compile** — and you must trigger it (decorator + 1st
compilation), over a subset of Python only.

## 4. Warmup, measured

We time **every call** of a sum, repeated in a loop. The **1st call** of the `numba`
function must **compile** (big spike); the time then **decays** to a native plateau. The
**pure** Python loop (never compiled) and `numpy` (ahead-of-time C) are **flat** — no spike.

This is the *measured* version of the "warmup": we'll always discard this 1st call before timing
(cf. Pitfall A below). *(The code producing these curves also lives in `experiences/warmup_jit/`.)*

In [ ]:
import time
import matplotlib.pyplot as plt

# dedicated array (10^6) so the pure loop stays practical over many calls
w = np.random.rand(1_000_000)
w_list = w.tolist()

@njit
def sum_njit_warm(x):        # FRESH function: its 1st call triggers compilation
    s = 0.0
    for i in range(x.shape[0]):
        s += x[i]
    return s

def per_call_times(fn, arg, reps):
    """Time (s) of each successive call of fn(arg)."""
    ts = []
    for _ in range(reps):
        t0 = time.perf_counter_ns()
        fn(arg)
        ts.append((time.perf_counter_ns() - t0) * 1e-9)
    return ts

R, R_pure = 400, 40
t_numba = per_call_times(sum_njit_warm, w, R)         # spike on 1st call (compilation)
t_numpy = per_call_times(np.sum, w, R)                # flat (compiled C)
t_pure  = per_call_times(mysum, w_list, R_pure)    # flat but slow (interpreted)

print(f"numba : 1st call = {t_numba[0]*1e3:8.2f} ms | steady = {np.median(t_numba[10:])*1e3:.3f} ms")
print(f"numpy : 1st call = {t_numpy[0]*1e3:8.3f} ms | steady = {np.median(t_numpy[10:])*1e3:.3f} ms")
print(f"pure  : 1st call = {t_pure[0]*1e3:8.2f} ms | steady = {np.median(t_pure[5:])*1e3:.2f} ms")

plt.figure(figsize=(8, 4.5))
plt.plot([t*1e3 for t in t_numba], ".", ms=3, label="numba njit")
plt.plot([t*1e3 for t in t_numpy], ".", ms=3, label="numpy")
plt.plot([t*1e3 for t in t_pure],  ".", ms=3, label="pure python")
plt.yscale("log")
plt.xlabel("iteration (call #)"); plt.ylabel("time / call (ms, log scale)")
plt.title("Warmup: decay of the sum time")
plt.legend(); plt.grid(True, which="both", alpha=0.3)
plt.tight_layout(); plt.show()

## 5. Measuring properly

- **Pitfall A — compilation.** The 1st `numba` call includes compilation: we exclude it (warm-up).
- **Pitfall B — a single measurement.** Repeat and take the **minimum** (`%timeit`, or our `best_time`).
- **Pitfall C — absolute seconds.** Think in **throughput**: here, bytes read / time.

In [ ]:
nbytes = v.nbytes
print(f"numpy.sum : {ms(t_np)}  |  throughput ≈ {nbytes/t_np/1e9:.1f} GB/s")
# %timeit np.sum(v)   # <- the standard tool in interactive Colab

## 6. Application: Monte-Carlo π

In [ ]:
import random
def pi_python(n):
    hits = 0
    for _ in range(n):
        x = random.random(); y = random.random()
        if x*x + y*y <= 1.0:
            hits += 1
    return 4*hits/n

def pi_numpy(n):
    x = np.random.rand(n); y = np.random.rand(n)
    return 4*np.count_nonzero(x*x + y*y <= 1.0)/n

@njit
def pi_numba(n):
    hits = 0
    for _ in range(n):
        x = np.random.random(); y = np.random.random()
        if x*x + y*y <= 1.0:
            hits += 1
    return 4*hits/n

n = 5_000_000
pi_numba(10)  # warm-up
print("π python :", ms(best_time(lambda: pi_python(n))))
print("π numpy  :", ms(best_time(lambda: pi_numpy(n))))
print("π numba  :", ms(best_time(lambda: pi_numba(n))))

## Wrap-up — to compare with Julia

| Variant | Time (to fill in) |
|---|---|
| pure Python loop | … |
| `numpy` | … |
| `numba njit` | … |
| **Julia (loop `@btime`)** | … |

> The pure Python loop is ~100× slower than numpy/numba/Julia, for code that *looks the same
> line for line*. Speed doesn't come from writing "the same" but from **leaving the interpreter**
> (numpy's C, numba's JIT) — which Julia does natively, everywhere.